# Stage 5.02 — freeze manifest and pairing

Run the 5A0 capability audit first. If the gate closes (the native horizon is exactly 8), Stage 5A and 5B manifests remain empty as required by the protocol. If the gate opens, the manifests are built from the audited allowed coverages.

In [ ]:
import json, os, subprocess
from pathlib import Path

R = Path.home() / "async-vla-latency-bench"
OUT = Path.home() / "stage5"
OFT = Path.home() / "openvla-oft"
PY = Path.home() / "venv-stage5-openvla/bin/python" if (Path.home() / "venv-stage5-openvla/bin/python").exists() else Path(os.environ.get("CONDA_PREFIX", sys.prefix)) / "bin/python"

prov = json.loads((OUT / "stage5_provenance.json").read_text())

# 5A0 static audit (no model load)
cmd = [
    str(PY if PY.exists() else Path(sys.executable).resolve()),
    "-m", "async_vla_benchmark.scripts.run_stage5a0",
    "--output-dir", str(OUT),
    "--openvla-oft-checkout", str(OFT),
    "--git-sha", prov["git_sha"],
    "--libero-plus-git-sha", prov["libero_plus_git_sha"],
]
print(subprocess.run(cmd, cwd=R, capture_output=True, text=True, check=True).stdout)


In [ ]:
import json
from pathlib import Path
OUT = Path.home() / "stage5"
audit = json.loads((OUT / "stage5_openvla_coverage_capability_audit.json").read_text())
print(audit["audit_conclusion"])
print("coverage_sweep_gt_native_allowed", audit["coverage_sweep_gt_native_allowed"])


In [ ]:
# Build conditional Stage 5A and 5B manifests.
import os, subprocess
from pathlib import Path

R = Path.home() / "async-vla-latency-bench"
OUT = Path.home() / "stage5"
prov = json.loads((OUT / "stage5_provenance.json").read_text())
PY = Path(sys.executable).resolve()

subprocess.run([
    str(PY), "-m", "async_vla_benchmark.scripts.make_stage5a_manifest",
    "--output", str(OUT / "stage5a_manifest.csv"),
    "--audit", str(OUT / "stage5_openvla_coverage_capability_audit.json"),
    "--git-sha", prov["git_sha"],
    "--libero-plus-git-sha", prov["libero_plus_git_sha"],
], cwd=R, check=True)

# 5B manifest is built from the 5A operating point file.
# If the gate closed, the selected operating point file is generated by the analysis step.
